# V01 — Plotly Express Foundations

**Plotly Express** is the high-level API. It covers 80% of real-world chart needs in 1–3 lines. Master it before touching `graph_objects`.

**Reference:** [Plotly Express docs](https://plotly.com/python/plotly-express/)

**Format:** Each exercise gives you a precise spec. Build the figure, pass the data assertions, then write a one-sentence business interpretation in the markdown cell below each chart.

**Allowed:** `plotly.express`, `pandas`, `numpy`


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.datasets import fetch_california_housing, fetch_openml

# --- Datasets ---
# California Housing
housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame.copy()
housing.columns = [c.lower() for c in housing.columns]
housing['price_category'] = pd.cut(
    housing['medhousval'],
    bins=[0, 1.5, 2.5, 3.5, 6],
    labels=['Low', 'Mid', 'High', 'Luxury']
)
housing['ocean_proximity'] = pd.cut(
    housing['longitude'],
    bins=[-125, -121, -118, -115],
    labels=['Coast', 'Central', 'Inland']
).astype(str)

# Online Retail
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()

# Monthly revenue summary
monthly = retail.groupby('Month').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('InvoiceNo', 'nunique'),
    Customers=('CustomerID', 'nunique')
).reset_index()

# Top countries
top_countries = (
    retail.groupby('Country')
    .agg(Revenue=('Revenue', 'sum'), Orders=('InvoiceNo', 'nunique'))
    .reset_index()
    .nlargest(15, 'Revenue')
)

print(f"Housing: {housing.shape} | Retail: {retail.shape}")
housing.head(3)

---
## Exercise 1 — Scatter Plot with Color & Size Encoding

**Spec:** Build a scatter plot of California housing data.
- X: `medinc` (median income)
- Y: `medhousval` (median house value)
- Color: `ocean_proximity`
- Size: `population` (cap display size range to [2, 20] using `size_max=20`)
- Opacity: 0.6
- Title: `'California Housing: Income vs Value by Location'`
- X-axis label: `'Median Income (10k USD)'`
- Y-axis label: `'Median House Value (100k USD)'`
- Use color sequence: `px.colors.qualitative.Set2`
- Assign to `fig1`

In [ ]:
# YOUR CODE HERE
fig1 = None

fig1.show()

In [ ]:
# --- ASSERTIONS ---
assert fig1 is not None
assert fig1.layout.title.text == 'California Housing: Income vs Value by Location'
assert fig1.layout.xaxis.title.text == 'Median Income (10k USD)'
assert fig1.layout.yaxis.title.text == 'Median House Value (100k USD)'
assert len(fig1.data) == housing['ocean_proximity'].nunique(), "One trace per ocean_proximity group"
# Size encoding present
assert fig1.data[0].marker.sizemode is not None or fig1.data[0].marker.size is not None
print("✓ Exercise 1 passed")

**Interpretation:** *(What pattern do you see? What does it suggest about location and housing prices?)*

---
## Exercise 2 — Line Chart with Multiple Series

**Spec:** Monthly revenue and order count trends as a dual-series line chart.
- X: `Month`
- Y: `Revenue`
- Add a second trace manually using `fig.add_scatter` for `Orders` on the same figure (normalized to [0, max(Revenue)] scale so both fit — compute `Orders_scaled = Orders / Orders.max() * Revenue.max()`)
- Markers on both lines: `mode='lines+markers'`
- Line for Revenue: color `'#1f77b4'`, width 2
- Line for Orders: color `'#ff7f0e'`, width 2, dash `'dot'`
- Title: `'Monthly Revenue & Order Volume'`
- Assign to `fig2`

In [ ]:
import plotly.graph_objects as go

monthly_plot = monthly.copy()
monthly_plot['Orders_scaled'] = (
    monthly_plot['Orders'] / monthly_plot['Orders'].max() * monthly_plot['Revenue'].max()
)

# YOUR CODE HERE
fig2 = None

fig2.show()

In [ ]:
# --- ASSERTIONS ---
assert fig2.layout.title.text == 'Monthly Revenue & Order Volume'
assert len(fig2.data) == 2, "Must have 2 traces"
trace_names = [t.name for t in fig2.data]
assert 'Revenue' in str(trace_names) or any('rev' in n.lower() for n in trace_names)
for trace in fig2.data:
    assert trace.mode == 'lines+markers', f"Trace {trace.name} must use lines+markers"
print("✓ Exercise 2 passed")

**Interpretation:** *(Describe the trend. Are revenue and orders moving together? Any anomalies?)*

---
## Exercise 3 — Horizontal Bar Chart with Sorting

**Spec:** Top 15 countries by revenue — horizontal bar chart.
- X: `Revenue`
- Y: `Country` — sorted by Revenue **ascending** (so largest is at top)
- Color: `Revenue` — use colorscale `'Blues'`
- Text on bars: Revenue formatted as `'$X,XXX'` — use `text_auto=False` and set `text` manually
- Title: `'Top 15 Countries by Revenue'`
- Remove colorbar (set `showscale=False` on the marker)
- Assign to `fig3`

In [ ]:
countries_sorted = top_countries.sort_values('Revenue', ascending=True)

# YOUR CODE HERE
fig3 = None

fig3.show()

In [ ]:
# --- ASSERTIONS ---
assert fig3.layout.title.text == 'Top 15 Countries by Revenue'
assert len(fig3.data[0].x) == 15
# Y-axis should be sorted: last item (top of chart) should be highest revenue
y_vals = list(fig3.data[0].y)
x_vals = list(fig3.data[0].x)
assert x_vals[-1] == max(x_vals), "Highest revenue country must be at top"
# Text labels present
assert fig3.data[0].text is not None and len(fig3.data[0].text) == 15
print("✓ Exercise 3 passed")

**Interpretation:** *(How concentrated is revenue across countries? What does this mean for market strategy?)*

---
## Exercise 4 — Histogram with Distribution Overlay

**Spec:** Distribution of `medhousval` in the housing dataset.
- `nbins=50`
- Color: `'#2196F3'`, opacity 0.75
- Add a vertical line (using `fig.add_vline`) at the **median** value: color `'red'`, dash `'dash'`, width 2
- Add a vertical line at the **mean**: color `'orange'`, dash `'dot'`, width 2
- Add annotations for both lines (label them `'Median'` and `'Mean'`)
- Title: `'Distribution of Median House Values'`
- X-axis label: `'Median House Value (100k USD)'`
- Assign to `fig4`

In [ ]:
median_val = housing['medhousval'].median()
mean_val = housing['medhousval'].mean()

# YOUR CODE HERE
fig4 = None

fig4.show()

In [ ]:
# --- ASSERTIONS ---
assert fig4.layout.title.text == 'Distribution of Median House Values'
assert fig4.layout.xaxis.title.text == 'Median House Value (100k USD)'
# Check vertical lines exist in layout shapes
shapes = fig4.layout.shapes
assert shapes is not None and len(shapes) >= 2, "Must have at least 2 vertical lines"
shape_x = [s.x0 for s in shapes]
assert any(abs(x - median_val) < 0.01 for x in shape_x), "Median line missing"
assert any(abs(x - mean_val) < 0.01 for x in shape_x), "Mean line missing"
print(f"✓ Exercise 4 passed — Median: {median_val:.2f}, Mean: {mean_val:.2f}")

**Interpretation:** *(Is the distribution skewed? What does the gap between mean and median tell you?)*

---
## Exercise 5 — Box Plot: Distribution Comparison

**Spec:** Compare `medhousval` distributions across `price_category` groups.
- X: `price_category`
- Y: `medhousval`
- Color: `price_category` — use `px.colors.sequential.Viridis` (4 colors)
- Show all points with `points='outliers'`
- Box width: `box` group ordering must be `['Low', 'Mid', 'High', 'Luxury']` left to right
- Title: `'House Value Distribution by Price Category'`
- Set `showlegend=False`
- Assign to `fig5`

In [ ]:
# YOUR CODE HERE
fig5 = None

fig5.show()

In [ ]:
# --- ASSERTIONS ---
assert fig5.layout.title.text == 'House Value Distribution by Price Category'
assert fig5.layout.showlegend == False
assert len(fig5.data) == 4, "One box per price category"
box_names = [t.name for t in fig5.data]
assert box_names == ['Low', 'Mid', 'High', 'Luxury'], f"Wrong order: {box_names}"
print("✓ Exercise 5 passed")

**Interpretation:** *(What do the box widths and whisker lengths tell you about each segment?)*

---
## Exercise 6 — Grouped Bar Chart

**Spec:** Revenue and order count by day of week.

First build `dow_summary`: group `retail` by `DayOfWeek`, compute `Revenue` (sum) and `Orders` (nunique InvoiceNo). Normalize both to 0–100 index (divide by max * 100) so they're on the same scale.

- Use `barmode='group'`
- Order x-axis: Monday through Sunday
- Colors: Revenue = `'#1976D2'`, Orders = `'#F57C00'`
- Add a horizontal reference line at y=100 (average): `add_hline(y=100, line_dash='dash', line_color='grey')`
- Title: `'Revenue & Orders Index by Day of Week (100 = Average)'`
- Assign to `fig6`

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# YOUR CODE HERE: build dow_summary, then fig6
dow_summary = None
fig6 = None

fig6.show()

In [ ]:
# --- ASSERTIONS ---
assert dow_summary is not None and len(dow_summary) == 7
assert fig6.layout.title.text == 'Revenue & Orders Index by Day of Week (100 = Average)'
assert fig6.layout.barmode == 'group'
assert len(fig6.data) == 2
# Check ordering
x_vals = list(fig6.data[0].x)
assert x_vals == day_order, f"Days not in correct order: {x_vals}"
# Check hline
shapes = fig6.layout.shapes
assert any(s.y0 == 100 for s in shapes), "Reference line at 100 missing"
print("✓ Exercise 6 passed")

**Interpretation:** *(Which day drives the most volume? Are revenue and orders always correlated by day?)*

---
## Exercise 7 — Scatter Matrix (Splom)

**Spec:** Explore pairwise relationships in housing data.
- Use `px.scatter_matrix`
- Dimensions: `['medinc', 'houseage', 'averooms', 'medhousval']`
- Color: `price_category`
- Diagonal: `'histogram'`
- Marker size: 2, opacity: 0.5
- Title: `'Housing Feature Relationships'`
- Assign to `fig7`

In [ ]:
# YOUR CODE HERE
fig7 = None

fig7.show()

In [ ]:
# --- ASSERTIONS ---
assert fig7.layout.title.text == 'Housing Feature Relationships'
dims = [d.label for d in fig7.data[0].dimensions]
assert set(dims) == {'medinc', 'houseage', 'averooms', 'medhousval'}
assert fig7.data[0].diagonal.visible == True
print("✓ Exercise 7 passed")

**Interpretation:** *(Which feature pair has the strongest visual correlation with house value? Any surprising relationships?)*

---
## Exercise 8 — Violin Plot with Embedded Box

**Spec:** Revenue distribution by day of week using violin plots.

Build `dow_revenue`: all individual transaction revenues per day (not aggregated).
- Use `px.violin`
- X: `DayOfWeek`, ordered Monday–Sunday
- Y: `Revenue`
- Color: `DayOfWeek`
- `box=True` (show embedded box plot)
- `points=False`
- Cap Y-axis at 500 (most revenue is < 500; outliers distort the violin)
- Title: `'Revenue Distribution by Day of Week'`
- `showlegend=False`
- Assign to `fig8`

In [ ]:
dow_revenue = retail[['DayOfWeek', 'Revenue']].copy()

# YOUR CODE HERE
fig8 = None

fig8.show()

In [ ]:
# --- ASSERTIONS ---
assert fig8.layout.title.text == 'Revenue Distribution by Day of Week'
assert fig8.layout.showlegend == False
assert fig8.layout.yaxis.range[1] == 500
assert len(fig8.data) == 7
print("✓ Exercise 8 passed")

**Interpretation:** *(Which days have the widest spread? What does the violin shape tell you about typical transaction sizes?)*

---
## Exercise 9 — Area Chart: Stacked Composition

**Spec:** Show how order volume and customer count evolve over time, stacked.

Melt `monthly` so that `Orders` and `Customers` are in a long-format DataFrame `monthly_long` with columns `Month`, `Metric`, `Value`.

- Use `px.area`
- X: `Month`, Y: `Value`, Color: `Metric`
- `groupnorm=''` (absolute, not normalized)
- Colors: Orders = `'#42A5F5'`, Customers = `'#66BB6A'`
- Add `line_shape='spline'`
- Title: `'Monthly Orders & Customers Over Time'`
- Assign to `fig9`

In [ ]:
monthly_long = monthly.melt(
    id_vars='Month',
    value_vars=['Orders', 'Customers'],
    var_name='Metric',
    value_name='Value'
)

# YOUR CODE HERE
fig9 = None

fig9.show()

In [ ]:
# --- ASSERTIONS ---
assert fig9.layout.title.text == 'Monthly Orders & Customers Over Time'
assert len(fig9.data) == 2
trace_names = {t.name for t in fig9.data}
assert trace_names == {'Orders', 'Customers'}
assert fig9.data[0].line.shape == 'spline'
print("✓ Exercise 9 passed")

**Interpretation:** *(Is customer count growing proportionally with orders? What does a gap suggest?)*

---
## Exercise 10 — Capstone: Executive KPI Chart

**Spec:** Build a combined KPI summary chart — a waterfall chart showing month-over-month revenue change.

1. Compute `mom_change`: month-over-month revenue delta from `monthly`. Columns: `Month`, `Revenue`, `MoM_Change`, `Type` (`'Increase'` / `'Decrease'` / `'Total'` for the last bar).
2. Use `go.Waterfall` to build the chart:
   - `measure`: list of `'relative'` for changes, `'total'` for the final bar
   - `x`: Month labels
   - `y`: MoM_Change values (last bar = total cumulative)
   - Increasing color: `'#26A69A'`, Decreasing: `'#EF5350'`, Total: `'#42A5F5'`
   - `connector_line_color`: `'#90A4AE'`
   - Title: `'Month-over-Month Revenue Change'`
3. Assign to `fig10`

In [ ]:
import plotly.graph_objects as go

# YOUR CODE HERE: build mom_change DataFrame and fig10 waterfall
mom_change = None
fig10 = None

fig10.show()

In [ ]:
# --- ASSERTIONS ---
assert fig10.layout.title.text == 'Month-over-Month Revenue Change'
assert len(fig10.data) == 1
assert fig10.data[0].type == 'waterfall'
measure = list(fig10.data[0].measure)
assert measure[-1] == 'total', "Last bar must be total"
assert all(m == 'relative' for m in measure[:-1]), "All others must be relative"
assert fig10.data[0].increasing.marker.color == '#26A69A'
assert fig10.data[0].decreasing.marker.color == '#EF5350'
print("✓ Exercise 10 passed")

**Interpretation:** *(Which months had the biggest swings? Is there a pattern — seasonal acceleration or deceleration?)*